This script fetches match data for the 1. and 2. Bundesliga (seasons 2020–2025) from the OpenLigaDB API, parses key match details (teams, goals, location, date), and filters for Holstein Kiel home games. Results are saved as a CSV file.

Developed with the assistance of [Claude (Anthropic)](https://claude.ai)

In [2]:
import requests
import polars as pl

seasons = [2020, 2021, 2022, 2023, 2024, 2025]
leagues = ["bl1", "bl2"]
all_matches = []

for season in seasons:
    for league in leagues:
        url = f"https://api.openligadb.de/getmatchdata/{league}/{season}"
        response = requests.get(url)
        if response.status_code != 200:
            continue
        data = response.json()
        if not data:
            continue

        for m in data:
            all_matches.append({
                "league":       league,
                "season":       season,
                "matchday":     m["group"]["groupOrderID"],
                "home":         m["team1"]["teamName"],
                "away":         m["team2"]["teamName"],
                "goals_home":   m["matchResults"][-1]["pointsTeam1"] if m["matchResults"] else None,
                "goals_away":   m["matchResults"][-1]["pointsTeam2"] if m["matchResults"] else None,
                "date":         m["matchDateTimeUTC"],
                "location_id":  m["location"]["locationID"]       if m["location"] else None,
                "city":         m["location"]["locationCity"]      if m["location"] else None,
                "stadium":      m["location"]["locationStadium"]   if m["location"] else None,
            })

df = pl.DataFrame(all_matches)

# Only Home Games
kiel_home = df.filter(pl.col("home").str.contains("Kiel"))

kiel_home.write_csv("../../data/Football/kiel_home_games.csv")